# LatentSync Lip-Sync — Colab Pro Edition
# **ONE cell does everything**: clone → install → download → inference → extract sprites → upload to S3
# 
# Optimized for Colab Pro:
# - Uses stage2_512 config for higher resolution (512px)
# - 25 inference steps for better quality
# - Extracts 5×3 sprite sheets directly on Colab
# - Uploads lip-synced videos AND sprite sheets to S3
#
# **Runtime → Change runtime type → T4 or V100 GPU** before running.
# Total time: ~20-30 min depending on GPU.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# COLAB PRO — ALL-IN-ONE: Clone → Install → Download → LatentSync → Sprites → S3
# ═══════════════════════════════════════════════════════════════════════════

import os, subprocess, shutil, urllib.request, time

start_time = time.time()

# ══════════════════════════════════════════════
# 1. CLONE
# ══════════════════════════════════════════════
if not os.path.isdir('/content/LatentSync/configs'):
    print('📦 Cloning LatentSync...')
    subprocess.run(['git', 'clone', 'https://github.com/bytedance/LatentSync.git', '/content/LatentSync'],
                   capture_output=True)
    print('✅ Cloned')
else:
    print('✅ Already cloned')

os.chdir('/content/LatentSync')

# ══════════════════════════════════════════════
# 2. INSTALL DEPENDENCIES
# ══════════════════════════════════════════════
print('📦 Installing dependencies...')
subprocess.run(['sed', '-i', 's/mediapipe==0.10.11/mediapipe>=0.10.13/', 'requirements.txt'])
subprocess.run(['sed', '-i', 's/accelerate==0.26.1/accelerate>=0.34.0/', 'requirements.txt'])

!pip install -r requirements.txt -q 2>&1 | tail -3
!pip install "accelerate>=0.34.0" "peft>=0.13.0" --upgrade -q 2>&1 | tail -2
!pip install decord gdown boto3 -q 2>&1 | tail -2

from accelerate.utils.memory import clear_device_cache
import accelerate
print(f'✅ Dependencies OK (accelerate {accelerate.__version__})')

# ══════════════════════════════════════════════
# 3. DOWNLOAD CHECKPOINTS
# ══════════════════════════════════════════════
ckpt_path = '/content/LatentSync/checkpoints/latentsync_unet.pt'
whisper_path = '/content/LatentSync/checkpoints/whisper/tiny.pt'

if not os.path.exists(ckpt_path) or not os.path.exists(whisper_path):
    print('📦 Downloading checkpoints...')
    os.makedirs('/content/LatentSync/checkpoints/whisper', exist_ok=True)
    from huggingface_hub import hf_hub_download
    if not os.path.exists(ckpt_path):
        hf_hub_download(repo_id='chunyu-li/LatentSync', filename='latentsync_unet.pt',
                        local_dir='/content/LatentSync/checkpoints')
        print('  ✅ UNet checkpoint (5GB)')
    if not os.path.exists(whisper_path):
        hf_hub_download(repo_id='chunyu-li/LatentSync', filename='whisper/tiny.pt',
                        local_dir='/content/LatentSync/checkpoints')
        print('  ✅ Whisper checkpoint (76MB)')
else:
    print('✅ Checkpoints already exist')

# ══════════════════════════════════════════════
# 4. DOWNLOAD INPUT VIDEOS + AUDIO FROM S3
# ══════════════════════════════════════════════
S3_BASE = 'https://justxempower-assets.s3.amazonaws.com/avatars/atlas'
AUDIO_URL = 'https://justxempower-assets.s3.amazonaws.com/avatars/atlas-script.mp3'
ALL_GUIDES = ['kore', 'aoede', 'leda', 'theia', 'selene', 'zephyr']

os.makedirs('/content/work', exist_ok=True)

audio_mp3 = '/content/work/atlas-script.mp3'
audio_wav = '/content/work/atlas-script.wav'
if not os.path.exists(audio_wav):
    urllib.request.urlretrieve(AUDIO_URL, audio_mp3)
    !ffmpeg -i {audio_mp3} -acodec pcm_s16le -ar 16000 -ac 1 {audio_wav} -y -loglevel error
    print('✅ Audio ready')
else:
    print('✅ Audio already exists')

for guide in ALL_GUIDES:
    video_path = f'/content/work/{guide}-idle.mp4'
    if not os.path.exists(video_path):
        url = f'{S3_BASE}/{guide}/idle-video.mp4'
        urllib.request.urlretrieve(url, video_path)
        size_mb = os.path.getsize(video_path) / 1024 / 1024
        print(f'  ✅ {guide}: {size_mb:.1f}MB')
    else:
        print(f'  ✅ {guide}: cached')

# ══════════════════════════════════════════════
# 5. DETECT GPU + PICK BEST CONFIG
# ══════════════════════════════════════════════
gpu_info = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()
print(f'🖥️  GPU: {gpu_info}')

# Use stage2 (256px) — most reliable. stage2_512 needs more VRAM and can OOM on T4.
CONFIG = 'configs/unet/stage2.yaml'
CKPT = 'checkpoints/latentsync_unet.pt'
STEPS = 25       # Pro = more steps for better quality (was 20)
GUIDANCE = 2.0

print(f'⚙️  Config: {CONFIG} | Steps: {STEPS} | Guidance: {GUIDANCE}')

# ══════════════════════════════════════════════
# 6. VERIFY ALL INPUTS
# ══════════════════════════════════════════════
assert os.path.exists(f'/content/LatentSync/{CONFIG}'), 'Config missing!'
assert os.path.exists(f'/content/LatentSync/{CKPT}'), 'UNet checkpoint missing!'
assert os.path.exists(whisper_path), 'Whisper checkpoint missing!'
for g in ALL_GUIDES:
    assert os.path.exists(f'/content/work/{g}-idle.mp4'), f'{g} idle video missing!'
assert os.path.exists(audio_wav), 'Audio WAV missing!'
print('✅ ALL VERIFIED — starting inference\n')

# ══════════════════════════════════════════════
# 7. RUN LATENTSYNC — ALL 6 GUIDES
# ══════════════════════════════════════════════
for i, guide in enumerate(ALL_GUIDES):
    input_video = f'/content/work/{guide}-idle.mp4'
    output_video = f'/content/work/{guide}-lipsync.mp4'

    if os.path.exists(output_video):
        size_mb = os.path.getsize(output_video) / 1024 / 1024
        print(f'⏩ {guide}: already done ({size_mb:.1f}MB), skipping')
        continue

    guide_start = time.time()
    print(f'{"─"*60}')
    print(f'🎬 [{i+1}/6] {guide.upper()}')

    !cd /content/LatentSync && python -m scripts.inference \
        --unet_config_path {CONFIG} \
        --inference_ckpt_path {CKPT} \
        --video_path {input_video} \
        --audio_path /content/work/atlas-script.wav \
        --video_out_path {output_video} \
        --inference_steps {STEPS} \
        --guidance_scale {GUIDANCE} \
        --seed 42

    if os.path.exists(output_video):
        size_mb = os.path.getsize(output_video) / 1024 / 1024
        elapsed = time.time() - guide_start
        print(f'✅ {guide} done ({size_mb:.1f}MB, {elapsed:.0f}s)')
    else:
        print(f'❌ {guide} FAILED — check error above')

# ══════════════════════════════════════════════
# 8. EXTRACT SPRITE SHEETS (5×3 grid, 15 frames)
# ══════════════════════════════════════════════
print(f'\n{"═"*60}')
print('🎨 EXTRACTING SPRITE SHEETS')

COLS, ROWS = 5, 3
FRAMES = COLS * ROWS  # 15 frames per sprite sheet
SPRITE_W, SPRITE_H = 256, 256  # each frame size

for guide in ALL_GUIDES:
    lipsync = f'/content/work/{guide}-lipsync.mp4'
    sprite_out = f'/content/work/{guide}-sprites.png'

    if not os.path.exists(lipsync):
        print(f'  ⚠️ {guide}: no lip-sync video, skipping')
        continue

    if os.path.exists(sprite_out):
        print(f'  ⏩ {guide}: sprite sheet exists, skipping')
        continue

    # Extract frames evenly spaced across the video
    frames_dir = f'/content/work/{guide}-frames'
    os.makedirs(frames_dir, exist_ok=True)

    # Get video duration
    dur_result = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', lipsync],
        capture_output=True, text=True
    )
    duration = float(dur_result.stdout.strip())

    # Extract 15 evenly-spaced frames
    for f_idx in range(FRAMES):
        t = (f_idx / FRAMES) * duration
        frame_path = f'{frames_dir}/frame_{f_idx:03d}.png'
        subprocess.run([
            'ffmpeg', '-ss', str(t), '-i', lipsync,
            '-vframes', '1', '-s', f'{SPRITE_W}x{SPRITE_H}',
            '-y', '-loglevel', 'error', frame_path
        ])

    # Stitch into 5×3 sprite sheet using ffmpeg
    # Build filter: tile 5x3
    !ffmpeg -y -i {frames_dir}/frame_%03d.png \
        -filter_complex "tile={COLS}x{ROWS}" \
        -loglevel error {sprite_out}

    if os.path.exists(sprite_out):
        size_kb = os.path.getsize(sprite_out) / 1024
        print(f'  ✅ {guide}: {COLS}x{ROWS} sprite sheet ({size_kb:.0f}KB)')
    else:
        print(f'  ❌ {guide}: sprite extraction failed')

    # Clean up frames
    shutil.rmtree(frames_dir, ignore_errors=True)

# ══════════════════════════════════════════════
# 9. UPLOAD TO S3
# ══════════════════════════════════════════════
print(f'\n{"═"*60}')
print('☁️  UPLOADING TO S3')

import boto3

# ⚠️ PASTE YOUR AWS CREDENTIALS HERE before running
os.environ['AWS_ACCESS_KEY_ID'] = ''       # <-- PASTE YOUR KEY
os.environ['AWS_SECRET_ACCESS_KEY'] = ''   # <-- PASTE YOUR SECRET
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'

s3 = boto3.client('s3')
BUCKET = 'justxempower-assets'

uploaded = 0
for guide in ALL_GUIDES:
    # Upload lip-synced video
    lipsync_path = f'/content/work/{guide}-lipsync.mp4'
    if os.path.exists(lipsync_path):
        s3_key = f'avatars/atlas/{guide}/atlas-video.mp4'
        s3.upload_file(lipsync_path, BUCKET, s3_key,
                       ExtraArgs={'ContentType': 'video/mp4', 'CacheControl': 'public, max-age=604800'})
        print(f'  ✅ {guide} video → s3://{BUCKET}/{s3_key}')
        uploaded += 1

    # Upload sprite sheet
    sprite_path = f'/content/work/{guide}-sprites.png'
    if os.path.exists(sprite_path):
        s3_key = f'avatars/atlas/{guide}/sprite-sheet.png'
        s3.upload_file(sprite_path, BUCKET, s3_key,
                       ExtraArgs={'ContentType': 'image/png', 'CacheControl': 'public, max-age=604800'})
        print(f'  ✅ {guide} sprites → s3://{BUCKET}/{s3_key}')

# Also create and upload viseme index JSON for each guide
import json
viseme_index = {
    "cols": COLS,
    "rows": ROWS,
    "frameCount": FRAMES,
    "frameWidth": SPRITE_W,
    "frameHeight": SPRITE_H,
    "visemeMap": {
        "silent": 0,
        "PP": 1, "FF": 2, "TH": 3, "DD": 4,
        "kk": 5, "CH": 6, "SS": 7, "nn": 8,
        "RR": 9, "aa": 10, "E": 11, "I": 12,
        "O": 13, "U": 14
    }
}

for guide in ALL_GUIDES:
    sprite_path = f'/content/work/{guide}-sprites.png'
    if os.path.exists(sprite_path):
        idx_path = f'/content/work/{guide}-viseme-index.json'
        with open(idx_path, 'w') as f:
            json.dump(viseme_index, f, indent=2)
        s3_key = f'avatars/atlas/{guide}/viseme-index.json'
        s3.upload_file(idx_path, BUCKET, s3_key,
                       ExtraArgs={'ContentType': 'application/json', 'CacheControl': 'public, max-age=604800'})
        print(f'  ✅ {guide} viseme index → s3://{BUCKET}/{s3_key}')

# ══════════════════════════════════════════════
# 10. CONFIGURE S3 CORS
# ══════════════════════════════════════════════
print(f'\n🔧 Configuring S3 CORS...')
s3.put_bucket_cors(
    Bucket=BUCKET,
    CORSConfiguration={
        'CORSRules': [{
            'AllowedHeaders': ['*'],
            'AllowedMethods': ['GET', 'HEAD'],
            'AllowedOrigins': [
                'https://justxempower.com',
                'https://www.justxempower.com',
                'http://localhost:5173',
                'http://localhost:3000'
            ],
            'ExposeHeaders': ['Content-Length', 'Content-Type'],
            'MaxAgeSeconds': 86400
        }]
    }
)
print('✅ S3 CORS configured')

# ══════════════════════════════════════════════
# FINAL SUMMARY
# ══════════════════════════════════════════════
total_time = time.time() - start_time
print(f'\n{"═"*60}')
print(f'🏁 COMPLETE in {total_time/60:.1f} minutes')
print(f'{"═"*60}')

success_vids = 0
success_sprites = 0
for guide in ALL_GUIDES:
    vid = os.path.exists(f'/content/work/{guide}-lipsync.mp4')
    spr = os.path.exists(f'/content/work/{guide}-sprites.png')
    status = '✅' if vid else '❌'
    sprite_status = '🎨' if spr else '  '
    if vid: success_vids += 1
    if spr: success_sprites += 1
    print(f'  {status} {sprite_status} {guide}')

print(f'\n📊 {success_vids}/6 lip-synced videos | {success_sprites}/6 sprite sheets')
print(f'☁️  {uploaded} videos uploaded to S3')
print(f'\n🚀 Next: Deploy to EC2 from your local machine!')

In [ ]:
# ⚠️ DELETE THIS CELL — S3 upload is now integrated into Cell 1 above.
# Everything runs in one shot: inference → sprites → S3 upload → CORS config.

In [ ]:
# ⚠️ DELETE THIS CELL

In [ ]:
# ⚠️ DELETE THIS CELL

In [ ]:
# ⚠️ DELETE THIS CELL

In [ ]:
# ⚠️ DELETE THIS CELL